In [ ]:
"""abr_tools.py - functions for the single-trial rat ABR project (MCB 493 final project).

Every function receives everything it needs as arguments (Lecture 15: a function in a
.py file cannot see the notebook). Import in the notebook with, e.g.
    from abr_tools import load_trials, preprocess

Data: Bieszczad & Gungor Aydin (2026), Zenodo doi:10.5281/zenodo.23066665 (CC-BY 4.0).

Course sources: filtering and padding (Lectures 4, 6), Welch spectrum (Lecture 5),
Morlet wavelets (Lecture 7), surrogates and the +1 p-value (Lecture 8), ERPs, fixed
windows and rejection (Lectures 9-10), raster/PSTH (Lecture 11), Fano factor and label
shuffles (Lecture 12), mean vector length R (Lecture 14), .py modules and sanity checks
(Lecture 15).
Other citable sources: numpy, pandas, matplotlib, scipy and mne documentation; Python
standard library (hashlib, re, pathlib, itertools); MAD x 1.4826 (Leys et al. 2013);
ITPC (Tallon-Baudry et al. 1996; Lachaux et al. 1999).
"""
import hashlib
import itertools
import re
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import signal


# ---------------------------------------------------------------------------
# Files
# ---------------------------------------------------------------------------
# The 26 CSVs in the Zenodo archive (21 unique recordings + 5 duplicate copies).
EXPECTED_FILES = [
    "Figure 2/APP_01_hh.csv", "Figure 2/APP_04_ss.csv", "Figure 2/APP_05_hh.csv",
    "Figure 2/APP_06_ss.csv", "Figure 2/APP_102_ss.csv", "Figure 2/APP_105-hh.csv",
    "Figure 3/APP_33_pretraining_ss.csv", "Figure 3/APP_35_pretraining_ss.csv",
    "Figure 3/APP_43_pretraining_ss.csv", "Figure 3/APP_44_pretraining_ss.csv",
    "Figure 3/APP_50_pretraining_hh.csv", "Figure 3/APP_53_pretraining_hh.csv",
    "Figure 4/APP_33_ss_post6weeks.csv", "Figure 4/APP_33_ss_post9months.csv",
    "Figure 4/APP_33_ss_posttraining.csv", "Figure 4/APP_33_ss_pretraining.csv",
    "Figure 4/APP_53_hh_posttraining.csv", "Figure 4/APP_53_hh_pretraining.csv",
    "Supplemental Figure 1/APP_105_hh.csv", "Supplemental Figure 1/PSEN1_LF_01_Female.csv",
    "Supplemental Figure 1/PSEN1_LF_01_Male.csv", "Supplemental Figure 1/PSEN1_LF_02_Female.csv",
    "Supplemental Figure 1/PSEN1_LF_02_Male.csv", "Supplemental Figure 1/PSEN1_wildtype.csv",
    "test_data/testsubject_hh.csv", "test_data/testsubject_ss.csv",
]


def missing_files(folder):
    """Expected CSVs that are not in `folder` (empty list = complete data)."""
    folder = Path(folder)
    return [f for f in EXPECTED_FILES if not (folder / f).is_file()]


def list_unique_files(folder):
    """All CSVs under `folder`, skipping exact duplicates (same MD5 checksum).
    The Zenodo zip holds five recordings twice under different names; the copy in a
    'Figure ...' folder is kept and the 'test_data' copy is skipped.
    Returns (kept_paths, list of (skipped, same_as))."""
    folder = Path(folder)
    files = [p for p in folder.rglob("*.csv") if "__MACOSX" not in p.parts]
    files.sort(key=lambda p: ("test_data" in p.parts, str(p)))
    seen, keep, dups = {}, [], []
    for p in files:
        h = hashlib.md5(p.read_bytes()).hexdigest()
        if h in seen:
            dups.append((str(p.relative_to(folder)), str(seen[h].relative_to(folder))))
        else:
            seen[h] = p
            keep.append(p)
    return keep, dups


def describe_file(name):
    """Rat, genotype, condition, sex and age cohort from this dataset's file names,
    e.g. 'Figure 4/APP_33_ss_posttraining.csv'. In the paper, App 'ss' = Swedish risk
    variant (AppS), 'hh' = humanized control (AppH); Figure 2 rats are aged (>12 months),
    Figure 3 rats are young (5 months)."""
    stem = Path(name).stem.lower()
    folder = str(Path(name).parent)
    info = {"folder": folder, "rat": Path(name).stem, "genotype": "unknown",
            "condition": "untrained", "sex": "",
            "cohort": {"Figure 2": "aged", "Figure 3": "young", "Figure 4": "young"}.get(folder, "")}
    m = re.match(r"app[_-]?(\d+)", stem)
    if m:
        info["rat"] = "APP_" + m.group(1)
        if re.search(r"(^|[_-])ss($|[_-])", stem):
            info["genotype"] = "AppS"
        elif re.search(r"(^|[_-])hh($|[_-])", stem):
            info["genotype"] = "AppH"
    elif stem.startswith("psen1"):
        info["genotype"] = "Psen1_LF" if "_lf_" in stem else "Psen1_WT"
    for key, cond in [("post9months", "post_9months"), ("post6weeks", "post_6weeks"),
                      ("posttraining", "post"), ("pretraining", "pre")]:
        if key in stem:
            info["condition"] = cond
            break
    if "female" in stem:
        info["sex"] = "F"
    elif "male" in stem:
        info["sex"] = "M"
    return info


def load_trials(path, n_samples):
    """One session as a (trials x samples) array in microvolts.
    Returns (data_uV, layout description, units description)."""
    df = pd.read_csv(path, header=None)
    df = df.apply(pd.to_numeric, errors="coerce")
    data = df.dropna(how="any").to_numpy()
    if data.shape[1] == n_samples:
        layout = "one trial per row"
    elif data.shape[0] == n_samples:
        data, layout = data.T, "one trial per column (transposed)"
    else:
        raise ValueError(f"{path}: shape {data.shape}, neither dimension is {n_samples}")
    if np.median(np.abs(data)) < 1e-3:          # values like 1e-7 are volts
        data, units = data * 1e6, "volts -> uV"
    else:
        units = "assumed already uV"
    return data, layout, units


# ---------------------------------------------------------------------------
# Preprocessing
# ---------------------------------------------------------------------------
def bandpass(trials, fs, lowcut, highcut, order, pad_ms):
    """Lecture 4 filter: butter(..., output='sos') + sosfiltfilt, applied to each trial
    after mean-centering and mirror-padding by pad_ms (pad, filter, unpad)."""
    sos = signal.butter(order, [lowcut, highcut], fs=fs, btype="bandpass", output="sos")
    pad = int(pad_ms / 1000 * fs)
    x = trials - trials.mean(axis=1, keepdims=True)
    x = np.pad(x, ((0, 0), (pad, pad)), mode="reflect")
    x = signal.sosfiltfilt(sos, x, axis=1)
    return x[:, pad:-pad]


def reject_peak_to_peak(trials, k):
    """Lecture 10 peak-to-peak detector; threshold = median + k * MAD (x1.4826) of the
    trials' peak-to-peak values. Returns (kept_trials, keep_mask, threshold_uV)."""
    ptp = trials.max(axis=1) - trials.min(axis=1)
    med = np.median(ptp)
    mad = 1.4826 * np.median(np.abs(ptp - med))
    threshold = med + k * mad
    keep = ptp <= threshold
    return trials[keep], keep, threshold


def preprocess(raw, fs, lowcut, highcut, order, pad_ms, downsample, reject_k):
    """bandpass -> keep every `downsample`-th sample -> peak-to-peak rejection.
    Downsampling after the low-pass avoids aliasing (Lecture 4; Lecture 15 slide 56).
    Returns (clean_trials, new_fs, keep_mask, threshold_uV)."""
    x = bandpass(raw, fs, lowcut, highcut, order, pad_ms)[:, ::downsample]
    x, keep, threshold = reject_peak_to_peak(x, reject_k)
    return x, fs / downsample, keep, threshold


def noise_of_average(trials, rng):
    """RMS of the +/- average: flipping the sign of a random half of the trials
    cancels the evoked response and leaves only the noise left in an average."""
    n = len(trials)
    s = rng.permutation(np.r_[np.ones(n // 2), -np.ones(n - n // 2)])
    return float(np.sqrt(np.mean((trials * s[:, None]).mean(axis=0) ** 2)))


def noise_vs_n(trials, ns, rng, n_rep=20):
    """Sanity check (Lecture 15, slides 55 and 61): noise of an average of n trials
    should fall as 1/sqrt(n), a slope of -0.5 on log-log axes.
    Returns (ns, mean noise for each n, fitted log-log slope)."""
    out = []
    for n in ns:
        out.append(np.mean([noise_of_average(trials[rng.choice(len(trials), n, replace=False)], rng)
                            for _ in range(n_rep)]))
    out = np.array(out)
    slope = np.polyfit(np.log10(ns), np.log10(out), 1)[0]
    return np.asarray(ns), out, float(slope)


# ---------------------------------------------------------------------------
# Peaks
# ---------------------------------------------------------------------------
def ms_to_index(ms, fs):
    return int(round(ms / 1000 * fs))


def find_peak_windows(avg, fs, search_ms, min_peak_size, half_ms, flank_ms=0.4):
    """Positive peaks of an average, with numpy only: a local maximum (rises in, falls
    out) that stands min_peak_size x the search-range span above the lowest point within
    flank_ms on either side. Each peak gets a fixed window center +/- half_ms."""
    i0, i1 = ms_to_index(search_ms[0], fs), ms_to_index(search_ms[1], fs)
    seg = avg[i0:i1]
    d = np.diff(seg)
    local_max = np.where((d[:-1] > 0) & (d[1:] <= 0))[0] + 1
    span = seg.max() - seg.min()
    w = ms_to_index(flank_ms, fs)
    windows = []
    for i in local_max:
        left = seg[max(i - w, 0): i].min()
        right = seg[i + 1: i + 1 + w].min() if i + 1 < len(seg) else seg[i]
        if seg[i] - max(left, right) >= min_peak_size * span:
            c = (i0 + i) / fs * 1000
            windows.append({"peak": f"P{len(windows) + 1}", "center_ms": round(c, 3),
                            "lo_ms": round(c - half_ms, 3), "hi_ms": round(c + half_ms, 3)})
    return windows


def measure_peaks(avg, fs, windows, trough_search_ms):
    """Per window: mean amplitude (primary; not biased by trial count, Lecture 10),
    peak amplitude and latency, and trough-to-peak amplitude (the paper's PW2 measure)."""
    rows = []
    for w in windows:
        lo, hi = ms_to_index(w["lo_ms"], fs), ms_to_index(w["hi_ms"], fs) + 1
        seg = avg[lo:hi]
        p = lo + int(np.argmax(seg))
        t0 = max(p - ms_to_index(trough_search_ms, fs), 0)
        rows.append({"peak": w["peak"], "mean_amp_uV": float(seg.mean()),
                     "peak_amp_uV": float(avg[p]), "peak_latency_ms": p / fs * 1000,
                     "trough_to_peak_uV": float(avg[p] - avg[t0:p + 1].min()),
                     "peak_on_window_edge": bool(p == lo or p == hi - 1)})
    return rows


# ---------------------------------------------------------------------------
# Single-trial timing: ITPC and jitter
# ---------------------------------------------------------------------------
def equal_n(a, b, rng):
    """Subsample the larger set so both have the same number of trials
    (ITPC and peak measures depend on trial count; Lecture 10)."""
    n = min(len(a), len(b))
    pick = lambda x: x[np.sort(rng.choice(len(x), n, replace=False))]
    return pick(a), pick(b)


def phase_vectors(trials, fs, freqs, n_cycles):
    """Unit phase vectors exp(i*phase) from Morlet wavelets, as in Lecture 7
    (tfr_array_morlet, output='complex', then np.angle). Shape (trials, freqs, times)."""
    from mne.time_frequency import tfr_array_morlet
    W = tfr_array_morlet(trials[:, np.newaxis, :], sfreq=fs, freqs=freqs,
                         n_cycles=n_cycles, output="complex")
    return np.exp(1j * np.angle(W[:, 0]))


def itpc(u):
    """ITPC = length of the mean unit vector across trials (Lecture 14's R):
    0 = random phase, 1 = every trial at the same phase."""
    return np.abs(u.mean(axis=0))


def itpc_masks(n_times, fs, freqs, n_cycles, windows, band):
    """valid: times at least half a wavelet (n_cycles/f/2) from either epoch edge
    (Lecture 7: edges are junk). summary: valid AND inside `band` AND inside a peak window."""
    t = np.arange(n_times) / fs * 1000
    half = n_cycles / np.asarray(freqs)[:, None] / 2 * 1000
    valid = (t[None, :] >= half) & (t[None, :] <= t[-1] - half)
    in_win = np.zeros(n_times, bool)
    for w in windows:
        in_win |= (t >= w["lo_ms"]) & (t <= w["hi_ms"])
    in_band = (np.asarray(freqs) >= band[0]) & (np.asarray(freqs) <= band[1])
    return valid, valid & in_band[:, None] & in_win[None, :]


def itpc_summary(u, summary_mask):
    return float(itpc(u)[summary_mask].mean())


def peak_latencies(trials, fs, windows, subavg, n_boot, seed):
    """Latency of each peak in n_boot sub-averages of `subavg` random trials.
    Single trials are too noisy for peak picking, so sub-averages are used.
    A fixed `seed` makes the result identical every run (fixes the run-to-run
    instability found in the first version)."""
    rng = np.random.default_rng(seed)
    idx = np.argsort(rng.random((n_boot, len(trials))), axis=1)[:, :subavg]
    avgs = trials[idx].mean(axis=1)                       # (n_boot, n_times)
    lat = np.empty((n_boot, len(windows)))
    for j, w in enumerate(windows):
        lo, hi = ms_to_index(w["lo_ms"], fs), ms_to_index(w["hi_ms"], fs) + 1
        lat[:, j] = (lo + np.argmax(avgs[:, lo:hi], axis=1)) / fs * 1000
    return lat


def jitter(trials, fs, windows, subavg, n_boot, seed):
    """Latency jitter = SD of sub-average peak latencies (ms), one value per peak.
    Jittered single trials make the average 'lower and wider' (Lecture 10)."""
    return peak_latencies(trials, fs, windows, subavg, n_boot, seed).std(axis=0)


# ---------------------------------------------------------------------------
# Statistics
# ---------------------------------------------------------------------------
def permutation_p(a, b, stat, n_perm, rng):
    """Two-sided label-shuffle test of stat(b) - stat(a) (Lecture 12), with
    p = (1 + #null >= observed) / (1 + n_perm) (Lecture 8).
    Returns (observed difference, p, null distribution)."""
    obs = stat(b) - stat(a)
    pooled = np.concatenate([a, b])
    null = np.empty(n_perm)
    for k in range(n_perm):
        idx = rng.permutation(len(pooled))
        null[k] = stat(pooled[idx[len(a):]]) - stat(pooled[idx[:len(a)]])
    p = (1 + np.sum(np.abs(null) >= abs(obs))) / (1 + n_perm)
    return float(obs), float(p), null


def split_half_diff(x, stat):
    """Split-half null: stat(second half) - stat(first half) of ONE session, i.e. how
    much the measure moves with no training in between (halves kept in time order)."""
    h = len(x) // 2
    return float(stat(x[h:2 * h]) - stat(x[:h]))


def cohort_relabelings(df):
    """Every way to assign the observed number of AppS labels within each age cohort."""
    options = []
    for _, g in df.groupby("cohort"):
        k = int((g.genotype == "AppS").sum())
        options.append([set(c) for c in itertools.combinations(g.index, k)])
    for combo in itertools.product(*options):
        yield set().union(*combo)


def exact_genotype_test(df, col):
    """Exact permutation test on ANIMAL labels, stratified by cohort (Lecture 12:
    'one animal per group: shuffle animals'). Statistic: mean(AppS) - mean(AppH).
    Returns (observed difference, exact two-sided p, number of relabelings)."""
    values = df[col].to_numpy()
    is_s = (df.genotype == "AppS").to_numpy()
    obs = values[is_s].mean() - values[~is_s].mean()
    null = []
    for s_idx in cohort_relabelings(df):
        mask = np.isin(df.index, list(s_idx))
        null.append(values[mask].mean() - values[~mask].mean())
    null = np.array(null)
    return float(obs), float(np.mean(np.abs(null) >= abs(obs) - 1e-12)), len(null)
